# One-hot feature audit

This notebook evaluates categorical features that may be encoded using one-hot or other categorical encoding. It reports row coverage and outlier-filtered average delay for planned time, day of week, line number, station name, train number, first station, and final destination.

Values with below-average support are excluded from the identifier summaries. For each feature, the threshold is the average number of rows per observed unique value.

In [6]:
from pathlib import Path

import pandas as pd

import sys
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from data_utils import load_monthly_data
from prep_job import add_first_station

DATA_DIR = PROJECT_ROOT / "data"
AUDIT_COLUMNS = [
    "station_name",
    "final_destination_station",
    "train_number",
    "line_number",
    "train_line_ride_id",
    "delay_in_min",
    "train_type",
    "train_line_station_num",
    "arrival_planned_time",
    "departure_planned_time",
]

data = load_monthly_data(
    DATA_DIR,
    AUDIT_COLUMNS,
    require_planned_time=False,
)
data["planned_time"] = data["arrival_planned_time"].fillna(
    data["departure_planned_time"]
)
data = data.dropna(subset=["planned_time"]).copy()
data = add_first_station(data)
delay_for_hour_average = data[
    data["delay_in_min"].between(-1, 180)
].copy()


## Planned departure hour distribution

The unified `planned_time` uses the planned arrival timestamp when available and the planned departure timestamp as a fallback. This table shows the percentage of analysis rows and the average delay for each planned hour, using the 24-hour clock. The average is calculated after removing delay outliers below -1 minute and above 180 minutes, and excludes missing delay values.

In [7]:
data["planned_hour"] = data["planned_time"].dt.hour
delay_for_hour_average = data[
    data["delay_in_min"].between(-1, 180)
].copy()
planned_hour_distribution = data.groupby("planned_hour").agg(
    rows=("planned_hour", "size"),
)
hourly_average_delay = delay_for_hour_average.groupby("planned_hour")[
    "delay_in_min"
].mean().rename("average_delay_min")
planned_hour_distribution = planned_hour_distribution.join(hourly_average_delay)
planned_hour_distribution["percent"] = (
    planned_hour_distribution["rows"] / len(data) * 100
).round(2)
planned_hour_distribution["average_delay_min"] = planned_hour_distribution[
    "average_delay_min"
].round(2)
planned_hour_distribution.index = planned_hour_distribution.index.map(
    lambda hour: f"{hour:02d}:00"
)
planned_hour_distribution.index.name = "planned_hour"
planned_hour_distribution

,rows,average_delay_min,percent
planned_hour,,,
00:00,193925,8.71,1.33
01:00,90536,8.05,0.62
02:00,29709,9.86,0.20
03:00,29187,6.57,0.20
04:00,182706,3.20,1.25
05:00,499408,3.26,3.42
06:00,766068,3.62,5.24
07:00,818102,4.24,5.60
08:00,778776,4.69,5.33


Looks lice it's a direct correlation which make sense as the longer train is going the more is chance for delay, looks like "daily reset" is about 3am, as minimum delay is at 4am and maximum at 2am and 3 am sits exactly between those hours and have avg delay out those 2. Overall I think we have enough rows for each hour and this data seem to be quite impactful, thus its make sense to unique hour for one hot encoding.

### Frequencies of string and identifier values

The following table reports the number of unique values for each string-like column. This includes categorical fields and identifiers such as `train_number` and `line_number`; the identifiers are not interpreted as continuous numeric variables.

In [8]:
identifier_columns = {"train_number", "line_number"}
string_columns = [
    column
    for column in data.columns
    if column in identifier_columns
    or pd.api.types.is_string_dtype(data[column])
]

string_summary = (
    data[string_columns]
    .nunique(dropna=True)
    .rename("unique_values")
    .rename_axis("column")
    .reset_index()
    .sort_values("unique_values")
    .reset_index(drop=True)
)

string_summary

,column,unique_values
0,train_type,3
1,source_file,9
2,line_number,137
3,final_destination_station,877
4,first_station,936
5,station_name,2551
6,train_number,12531
7,train_line_ride_id,32261


looks like we definetly should drop train_line_ride_id as its just a unique identifier,

## Planned day-of-week distribution

This table uses the same unified `planned_time` as the hourly analysis. It reports the percentage of rows and the average delay for each day of the week. Delay values below -1 minute and above 180 minutes are excluded from the average.

In [9]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]
data["planned_day_of_week"] = pd.Categorical(
    data["planned_time"].dt.day_name(),
    categories=weekday_order,
    ordered=True,
)
planned_day_distribution = data.groupby(
    "planned_day_of_week", observed=False
).agg(rows=("planned_day_of_week", "size"))
weekday_average_delay = delay_for_hour_average.groupby(
    data.loc[delay_for_hour_average.index, "planned_day_of_week"]
)["delay_in_min"].mean().rename("average_delay_min")
planned_day_distribution = planned_day_distribution.join(weekday_average_delay)
planned_day_distribution["percent"] = (
    planned_day_distribution["rows"] / len(data) * 100
).round(2)
planned_day_distribution["average_delay_min"] = planned_day_distribution[
    "average_delay_min"
].round(2)
planned_day_distribution.index.name = "planned_day_of_week"
planned_day_distribution

,rows,average_delay_min,percent
planned_day_of_week,,,
Monday,2127807,5.93,14.56
Tuesday,2173131,6.16,14.87
Wednesday,2148105,6.33,14.70
Thursday,2154746,6.21,14.74
Friday,2153363,6.51,14.73
Saturday,1966073,5.64,13.45
Sunday,1892514,4.90,12.95


Looks like it does have an impact though not that big, it's clear that delay raise from Sunday till Friday and drops from Friday till Sunday; given that only 7 columns for one hot i believe its worth it.

## Delay by line, station, train, and route

The following tables separately report the number and percentage of rows, together with the average delay, for `line_number`, `station_name`, `train_number`, `first_station`, and `final_destination_station`. For each feature, values with fewer rows than the average rows per unique value are excluded. Missing values are excluded. The average delay excludes outliers below -1 minute or above 180 minutes.

In [10]:
identifier_columns = [
    "line_number",
    "station_name",
    "train_number",
    "first_station",
    "final_destination_station",
]

for column in identifier_columns:
    analysis_data = data
    if column in {"line_number", "first_station", "final_destination_station"}:
        analysis_data = data.dropna(subset=[column])

    values = analysis_data[column].astype("string")
    summary = values.value_counts().rename("rows").to_frame()
    average_rows_per_value = len(analysis_data) / len(summary)
    summary = summary[summary["rows"] >= average_rows_per_value]
    values = values.where(values.isin(summary.index))
    valid_delay = delay_for_hour_average.index.intersection(analysis_data.index)
    average_delay = analysis_data.loc[valid_delay].assign(
        _identifier=values.loc[valid_delay]
    ).groupby("_identifier")["delay_in_min"].mean().rename(
        "average_delay_min"
    )
    summary = summary.join(average_delay)
    summary["percent"] = (summary["rows"] / len(analysis_data) * 100).round(2)
    summary["average_delay_min"] = summary["average_delay_min"].round(2)
    summary.index.name = column

    print(
        f"{column} ({len(analysis_data):,} rows; "
        f"average rows per unique value: {average_rows_per_value:,.2f})"
    )
    display(summary)

line_number (12,367,458 rows; average rows per unique value: 90,273.42)


,rows,average_delay_min,percent
line_number,,,
RE7,996070,4.15,8.05
RE2,822602,6.46,6.65
RE3,692036,3.60,5.6
RE1,632394,6.26,5.11
RE4,506814,4.20,4.1
RE6,453607,4.71,3.67
RE30,408845,4.68,3.31
RE8,408205,4.52,3.3
RE50,374585,6.35,3.03


station_name (14,615,739 rows; average rows per unique value: 5,729.42)


,rows,average_delay_min,percent
station_name,,,
Berlin Hauptbahnhof,157173,7.94,1.08
Frankfurt (Main) Hbf,132530,9.72,0.91
Nürnberg Hbf,128445,7.62,0.88
Berlin Südkreuz,112547,7.16,0.77
Stuttgart Hbf,111652,7.45,0.76
...,...,...,...
Linda (Elster),5901,3.08,0.04
Kaufering,5836,7.11,0.04
Schönebeck-Frohse,5759,5.89,0.04


train_number (14,615,739 rows; average rows per unique value: 1,166.37)


,rows,average_delay_min,percent
train_number,,,
3311,10894,4.35,0.07
3309,10887,3.84,0.07
3307,10869,4.33,0.07
3305,10847,2.79,0.07
3313,10843,3.88,0.07
...,...,...,...
19369,1170,4.69,0.01
11193,1170,3.80,0.01
3701,1168,3.91,0.01


first_station (14,175,979 rows; average rows per unique value: 15,145.28)


,rows,average_delay_min,percent
first_station,,,
Frankfurt (Main) Hbf,530572,8.20,3.74
Nürnberg Hbf,524989,4.93,3.7
München Hbf,471171,11.47,3.32
Stuttgart Hbf,368929,7.05,2.6
Karlsruhe Hbf,328545,5.50,2.32
...,...,...,...
Wendlingen (Neckar),15636,2.93,0.11
Plochingen,15543,2.52,0.11
Hamburg-Harburg,15387,11.73,0.11


final_destination_station (13,294,125 rows; average rows per unique value: 15,158.64)


,rows,average_delay_min,percent
final_destination_station,,,
Frankfurt(Main)Hbf,511882,6.45,3.85
Nürnberg Hbf,477466,4.59,3.59
München Hbf,411156,11.01,3.09
Stuttgart Hbf,344517,5.58,2.59
Karlsruhe Hbf,306100,5.31,2.3
...,...,...,...
Wesel,15881,5.52,0.12
Löhne(Westf),15548,2.31,0.12
Winterberg(Westf),15389,4.88,0.12


At first glance line number might be useful, but same line numbers are used for lines in different regions thus it's not a valid feature and will be dropped. Also, it looks like it make sense to use stations and train number after applying thresholds for min row count, question is that what thresholds should be and what do we do with rows that under threshold.

## One-hot feature summary

The audit supports the following preprocessing and encoding design:

- `train_line_station_num` must be retained as a positive integer feature. It represents the train's position in the line, and later positions may have accumulated more delay.
- The categorical features to encode using one-hot encoding are `train_type`, `day_of_week`, planned hour, `first_station`, the current `station_name`, `final_destination_station`, and `train_number`.
- The average number of rows per unique value is a reasonable starting threshold for excluding sparse values before one-hot encoding. This threshold is calculated separately for each categorical feature. It is an initial rule and should be revisited using validation results and the resulting feature dimensionality.
- The target is created only from `delay_in_min` and `arrival_is_canceled`. `arrival_is_canceled` identifies cancellations, while `delay_in_min` supplies the delay class for non-cancelled rows.
- Other raw source fields must be dropped after the required derived features and target have been created. This includes technical identifiers such as `eva`, `source_file`, and `train_line_ride_id`, raw schedule columns after `planned_time` has been created, and helper fields that are not part of the selected feature set.

The resulting modelling data should therefore contain the selected encoded categorical features, the numeric `train_line_station_num`, and the target. No raw identifier or unused source column should be passed to the model.